In [13]:
    from PIL import Image
    from PIL.ExifTags import TAGS, GPSTAGS
    import math

    def get_exif(filename):
        image = Image.open(filename)
        image.verify()
        return image._getexif()

    def get_geotagging(exif):
        if not exif:
            raise ValueError("No EXIF metadata found")
        geotagging = {}
        for (idx, tag) in TAGS.items():
            if tag == 'GPSInfo':
                if idx not in exif:
                    raise ValueError("No EXIF geotagging found")
                for (key, val) in GPSTAGS.items():
                    if key in exif[idx]:
                        geotagging[val] = exif[idx][key]
        return geotagging

    def get_decimal_from_dms(dms, ref):
        degrees = dms[0].numerator / dms[0].denominator
        minutes = dms[1].numerator / dms[1].denominator / 60.0
        seconds = dms[2].numerator / dms[2].denominator / 3600.0

        if ref in ['S', 'W']:
            degrees = -degrees
            minutes = -minutes
            seconds = -seconds

        return degrees + minutes + seconds


    def get_decimal_from_exif_ratio(ratio):
        """Helper function to convert IFDRational to a float."""
        return ratio.numerator / ratio.denominator

    def calculate_new_coordinates(img_path, px_x, px_y):
        exif = get_exif(img_path)
        geotags = get_geotagging(exif)
        lat = get_decimal_from_dms(geotags['GPSLatitude'], geotags['GPSLatitudeRef'])
        lon = get_decimal_from_dms(geotags['GPSLongitude'], geotags['GPSLongitudeRef'])
        
        altitude = get_decimal_from_exif_ratio(geotags['GPSAltitude'])
        image_width, image_height = 1600, 1300  
        drone_altitude_meters = 120  # in meters
        focal_length_mm = 24  
        sensor_width_mm = 36  
        
        
        fov = 2 * math.atan((sensor_width_mm / 2) / focal_length_mm)
        scale = (2 * drone_altitude_meters * math.tan(fov / 2)) / image_width  
        
        # Calculate pixel distance from center
        center_x, center_y = image_width / 2, image_height / 2
        delta_x = (px_x - center_x) * scale
        delta_y = (px_y - center_y) * scale  
        
        # Convert delta_x and delta_y to degrees
        earth_radius_m = 6378137  # meters
        delta_lat = (delta_y / earth_radius_m) * (180 / math.pi)
        delta_lon = (delta_x / (earth_radius_m * math.cos(math.pi * lat / 180))) * (180 / math.pi)
        
        new_lat = lat + delta_lat
        new_lon = lon + delta_lon
        
        return new_lat, new_lon

    
    img_path = '/home/pranav/Mango_Orchard/100FPLAN/DJI_0100.JPG'
    pixel_x, pixel_y = 100, 90  
    new_lat, new_lon = calculate_new_coordinates(img_path, pixel_x, pixel_y)
    print(f"New latitude: {new_lat}, New longitude: {new_lon}")


New latitude: 28.439510172482116, New longitude: 80.92686906761243
